In [81]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import torchmetrics

In [82]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
train_df = pd.read_parquet("hf://datasets/FelixCS3/statsbomb-open-data-shots/" + splits["train"])
validation_df = pd.read_parquet("hf://datasets/FelixCS3/statsbomb-open-data-shots/" + splits["validation"])
test_df = pd.read_parquet("hf://datasets/FelixCS3/statsbomb-open-data-shots/" + splits["test"])

In [83]:
## Preprocessing

continuous_cols = ['distance_to_goal', 'angle_to_goal', 'x', 'y']
discrete_cols = ['under_pressure', 'is_goal']
categorical_cols = ['shot_type_name']

## under_pressure column already boolean
train_df["under_pressure"] = train_df["under_pressure"].astype(float)
validation_df["under_pressure"] = validation_df["under_pressure"].astype(float)
test_df["under_pressure"] = test_df["under_pressure"].astype(float)

preprocessor = ColumnTransformer(
    transformers=[
        ('continuous', StandardScaler(), continuous_cols),
        ('discrete', 'passthrough', discrete_cols),
        ('categorical', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ]
)

preprocessor.set_output(transform="pandas")
train_df = preprocessor.fit_transform(train_df)
val_df = preprocessor.transform(validation_df)
test_df = preprocessor.transform(test_df)

In [84]:
train_df

,continuous__distance_to_goal,continuous__angle_to_goal,continuous__x,continuous__y,discrete__under_pressure,discrete__is_goal,categorical__shot_type_name_Corner,categorical__shot_type_name_Free Kick,categorical__shot_type_name_Open Play,categorical__shot_type_name_Penalty
0,-1.389727,-0.017919,1.157030,0.349166,0.0,0,0.0,0.0,1.0,0.0
1,0.513920,0.493088,-0.286872,-1.413945,0.0,0,0.0,0.0,1.0,0.0
2,0.047995,0.075318,-0.082224,-0.937702,0.0,0,0.0,0.0,1.0,0.0
3,-0.827265,-1.442656,0.508980,0.035048,0.0,1,0.0,0.0,0.0,1.0
4,0.784327,-1.398003,-1.094093,-0.035881,0.0,0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
70421,-1.055398,0.835487,1.043337,0.713947,1.0,0,0.0,0.0,1.0,0.0
70422,1.470789,-0.699487,-1.662558,-0.785710,0.0,0,0.0,0.0,1.0,0.0
70423,0.727221,-1.396627,-1.037246,0.085713,1.0,0,0.0,0.0,1.0,0.0
70424,-0.739700,0.803458,0.804582,0.896338,0.0,0,0.0,0.0,1.0,0.0


In [85]:
feature_cols = ['continuous__distance_to_goal', 'continuous__angle_to_goal', 'discrete__under_pressure']

In [86]:
class XGDataset(Dataset):
    def __init__(self, df, feature_cols):
        self.x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.y = torch.tensor(df["discrete__is_goal"].values, dtype=torch.float32)
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [87]:
train_loader = DataLoader(XGDataset(train_df, feature_cols), batch_size=32, shuffle=True)
test_loader = DataLoader(XGDataset(test_df, feature_cols), batch_size=32, shuffle=False)

model = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)



lr = 0.001
epochs = 10

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):

    print(f"Epoch {epoch}")
    train_loss = 0

    for x, y in train_loader:
        y = y.float().view(-1, 1)

        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Training loss: {train_loss / len(train_loader)}")


from torchmetrics.classification import BinaryAUROC

model.eval()
test_loss = 0
auroc_metric = BinaryAUROC()

with torch.no_grad():
    for x, y in test_loader:
        logits = model(x)
        y = y.float().view(-1, 1) 
        y_prob = torch.sigmoid(logits)
        loss = loss_fn(logits, y)
        test_loss += loss.item()
        auroc_metric.update(y_prob, y)

print(f"Test loss: {test_loss / len(test_loader)}")
print(f"Test AUROC: {auroc_metric.compute()}")

Epoch 0
Training loss: 0.32158891319795285
Epoch 1
Training loss: 0.3021994834086235
Epoch 2
Training loss: 0.30161107316958474
Epoch 3
Training loss: 0.30105605677129027
Epoch 4
Training loss: 0.30080080374919627
Epoch 5
Training loss: 0.30043086944751335
Epoch 6
Training loss: 0.3001618509243045
Epoch 7
Training loss: 0.2998366894835995
Epoch 8
Training loss: 0.2997522595076602
Epoch 9
Training loss: 0.2994567955463705
Test loss: 0.30233519794284436
Test AUROC: 0.7573232650756836


In [89]:
## Model with expanded feature set, including xy coordinates and shot_type_name

expanded_features = ['continuous__distance_to_goal', 'continuous__angle_to_goal', 'discrete__under_pressure', 'continuous__x', 'continuous__y',]
expanded_features += [col for col in train_df.columns if str(col).startswith('categorical__shot_type_name')]

train_loader = DataLoader(XGDataset(train_df, expanded_features), batch_size=32, shuffle=True)
test_loader = DataLoader(XGDataset(test_df, expanded_features), batch_size=32, shuffle=False)

model = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)


lr = 0.001
epochs = 10

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

lr = 0.001
epochs = 10

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):

    print(f"Epoch {epoch}")
    train_loss = 0

    for x, y in train_loader:
        y = y.float().view(-1, 1)

        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Training loss: {train_loss / len(train_loader)}")


from torchmetrics.classification import BinaryAUROC

model.eval()
test_loss = 0
auroc_metric = BinaryAUROC()

with torch.no_grad():
    for x, y in test_loader:
        logits = model(x)
        y = y.float().view(-1, 1) 
        y_prob = torch.sigmoid(logits)
        loss = loss_fn(logits, y)
        test_loss += loss.item()
        auroc_metric.update(y_prob, y)

print(f"Test loss: {test_loss / len(test_loader)}")
print(f"Test AUROC: {auroc_metric.compute()}")

Epoch 0
Training loss: 0.3157990684547461
Epoch 1
Training loss: 0.2945570818968168
Epoch 2
Training loss: 0.2938434764885025
Epoch 3
Training loss: 0.2933647451665877
Epoch 4
Training loss: 0.2930205419070609
Epoch 5
Training loss: 0.29287457830530306
Epoch 6
Training loss: 0.2927147894759224
Epoch 7
Training loss: 0.2925723359726711
Epoch 8
Training loss: 0.29258785156811545
Epoch 9
Training loss: 0.29237041079093523
Test loss: 0.2952803517624304
Test AUROC: 0.7670670747756958
